# Grid Network Analysis — 3. Geographic and Business Intelligence Analysis

Builds the interactive Folium map (exported as standalone HTML for the
dashboard/report) and runs the business-intelligence proxy metrics from
`../src/bi.py`.

**Every BI metric below is an explicit proxy** — this synthetic dataset has
no real load, population, or demand data, so "underserved region" and
"capacity flag" are structural stand-ins, not verified operational findings.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
from src.cleaning import load_raw, clean_and_validate, merge_all
from src.geo import build_folium_map
from src.bi import utility_footprint, capacity_flags, underserved_regions, asset_age_profile

DATA_DIR = Path("../data")
OUTPUTS_DIR = Path("../outputs")

utilities, substations, lines = load_raw(DATA_DIR)
utilities, substations, lines, _ = clean_and_validate(utilities, substations, lines)
merged = merge_all(utilities, substations, lines)

## Geographic map

Substations are colored by voltage tier; lines are drawn as polylines
colored to match. Toggle layers with the control in the top-right corner.

In [2]:
grid_map = build_folium_map(substations, lines)
grid_map.save(str(OUTPUTS_DIR / "substation_map.html"))
grid_map

## Business intelligence (proxy metrics)

### Utility footprint by region
Line count per utility per source region — a proxy for infrastructure
footprint, not a measure of actual power delivered.

In [3]:
utility_footprint(merged).head(15)

,Utility Alias,Source Region,Line Count
13,GRIDCo,Greater Accra,5
21,NEDCo,Greater Accra,4
19,NEDCo,Central,3
18,NEDCo,Ashanti,3
17,GRIDCo,Western,3
16,GRIDCo,Volta,3
12,GRIDCo,Eastern,3
10,GRIDCo,Bono,3
8,ECG,Western,2
9,GRIDCo,Ashanti,2


### Capacity flags

Substations whose rated capacity sits in the bottom or top quartile for
their voltage tier. Flagged as "upgrade candidate" / "high capacity" —
without real load data, this is the closest available proxy for
over/under-provisioning.

In [4]:
capacity_flags(substations).sort_values("capacity_flag")

,Substation ID,Name,Region,Voltage (kV),Capacity (MVA),capacity_flag
7,8,Ejisu Substation,Ashanti,330,355.9,High capacity
8,9,Obuasi Substation,Ashanti,33,47.5,High capacity
17,18,Kasoa Substation,Central,11,52.4,High capacity
21,22,Nkawkaw Substation,Eastern,69,389.2,High capacity
22,23,Suhum Substation,Eastern,69,339.0,High capacity
23,24,Ho Substation,Volta,330,382.1,High capacity
26,27,Sogakope Substation,Volta,11,58.8,High capacity
28,29,Techiman Substation,Bono,33,44.2,High capacity
31,32,Yendi Substation,Northern,11,52.3,High capacity
33,34,Bolgatanga Substation,Upper East,161,325.9,High capacity


### Underserved regions (proxy)

Regions with both substation count and total rated capacity below the
cross-region median. A real assessment would need population and demand
data; this is a structural signal only.

In [5]:
underserved_regions(substations)

,Region,substation_count,total_capacity_mva,underserved_proxy
1,Benin,1,487.6,False
3,Burkina Faso,1,445.9,False
4,Burkina Faso border,1,156.1,True
15,Upper West,1,27.1,True
6,Cote d'Ivoire,1,262.6,True
7,Cote d'Ivoire border,1,285.1,True
13,Togo border,1,423.2,False
10,Guinea,1,251.6,True
12,Togo,1,120.2,True
14,Upper East,2,520.7,False


### Asset age profile

Commissioning years bucketed into decades, cross-tabbed with region and
status — a proxy for where older (higher fault-risk) infrastructure is
concentrated.

In [6]:
asset_age_profile(substations)

Region Ashanti  Benin   Bono Burkina Faso Burkina Faso border Central  \
Status  Active Active Active       Active              Active  Active   
decade                                                                  
1960s        0      0      0            0                   0       0   
1970s        0      0      0            0                   0       1   
1980s        1      0      0            1                   0       0   
1990s        1      1      1            0                   1       1   
2000s        2      0      1            0                   0       2   
2010s        1      0      0            0                   0       0   
2020s        0      0      1            0                   0       0   

Region Cote d'Ivoire Cote d'Ivoire border Eastern Greater Accra Guinea  \
Status        Active               Active  Active        Active Active   
decade                                                                   
1960s              0                    0       0             0      0   
1970s              0                    0       1             2      0   
1980s              0                    1       0             0      0   
1990s              0                    0       0             1      0   
2000s              0                    0       1             2      1   
2010s              1                    0       1             1      0   
2020s              0                    0       1             0      0   

Region Northern            Togo Togo border Upper East Upper West  Volta  \
Status   Active Inactive Active      Active     Active     Active Active   
decade                                                                     
1960s         0        1      0           0          0          0      0   
1970s         1        0      0           0          1          1      0   
1980s         0        0      0           0          0          0      1   
1990s         0        0      0           0          1          0      1   
2000s         0        0      0           0          0          0      1   
2010s         1        0      1           1          0          0      1   
2020s         0        0      0           0          0          0      0   

Region Western  
Status  Active  
decade          
1960s        1  
1970s        1  
1980s        0  
1990s        0  
2000s        1  
2010s        1  
2020s        0

## Limitations

All coordinates, capacities, commissioning years, and connections in this
dataset are synthetic and illustrative (generated by a seeded script) and
must not be presented as verified measurements or descriptions of Ghana's
actual electricity infrastructure. The BI metrics above are structural
proxies chosen because no real load, population, or demand data exists for
this coursework dataset — see `../report.md` for the full discussion.